# Lecture 08: Automatic Differentiation & Physics-Informed Neural Networks

**Course**: Machine Learning Applications in Physics (PHYG004, 2026 Spring)
**Instructor**: Prof. Young Woo Choi · Sogang University

**Data type**: Toy / Synthetic (analytic potentials, synthetic observations)

---

## Overview

| Mission | Topic | Estimated Time | Difficulty |
|---------|-------|---------------|------------|
| Theory | Autodiff, PINN, inverse problems | 40 min | Reading |
| A | JAX autodiff: `grad`, `hessian`, `vmap` | 25 min | Easy |
| B | PINN for the Schrödinger equation (QHO) | 30 min | Medium |
| B-ext1 | Anharmonic potential PINN (graded) | 15 min | Medium |
| B-ext2 | Excited-state PINN with orthogonality (graded) | 15 min | Hard |
| C | PINN inverse problem: recover V(x) coefficients | 25 min | Hard |

> **Google Colab**: Runtime → Change runtime type → CPU is sufficient for all missions.
> GPU is optional and will only marginally speed up the small networks used here.


In [ ]:
# Cell 1 — Install dependencies (run once on Colab)
!pip install jax jaxlib optax matplotlib --quiet


In [ ]:
# Cell 2 — Imports and setup
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
import numpy as np

# Reproducibility
key = jax.random.PRNGKey(42)

# Device check
print("JAX version:", jax.__version__)
print("Devices:", jax.devices())


---

## Part 1: Automatic Differentiation (Theory)

### Derivatives in Physics

Almost every quantity physicists care about involves a derivative:

$$\mathbf{F} = -\nabla V, \qquad \frac{d}{dt}\frac{\partial L}{\partial \dot{q}} = \frac{\partial L}{\partial q}, \qquad \chi = -\frac{\partial^2 F}{\partial h^2}$$

Three ways to compute derivatives on a computer:

| Method | Exact? | Cost | Limitation |
|--------|--------|------|------------|
| Symbolic | Yes | High | Expression swell — grows exponentially |
| Finite difference | No | Low | Step-size dilemma (truncation vs cancellation) |
| **Autodiff** | **Yes** | **Low** | Needs a differentiable framework |

### Why Not Finite Differences?

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

Large $h$ → truncation error $O(h^2)$. Small $h$ → floating-point cancellation error. No universally good $h$.

### How Autodiff Works: Computational Graphs

Consider $e = (a + b) \times (b + 1)$. Break into elementary steps:

$$c = a + b, \quad d = b + 1, \quad e = c \times d$$

Each edge in the computation graph carries a local partial derivative.
The chain rule sums over all paths:

$$\frac{\partial e}{\partial b} = \frac{\partial e}{\partial c}\frac{\partial c}{\partial b} + \frac{\partial e}{\partial d}\frac{\partial d}{\partial b} = d \cdot 1 + c \cdot 1$$

**Forward mode**: fix one input, push derivative forward. One pass → $\partial(\text{output})/\partial(\text{one input})$.

**Reverse mode** (= backpropagation): start from output, pull backward. One pass → $\partial(\text{one output})/\partial(\text{all inputs})$.

> For ML: minimize scalar loss $L(\theta)$ with $\theta \sim 10^6$ parameters.
> Reverse mode gives all $n$ gradients in **one backward pass**.

### Autodiff in JAX

```python
jax.grad(f)                    # scalar → scalar gradient
jax.grad(jax.grad(f))          # second derivative
jax.hessian(f)                 # full Hessian matrix
jax.vmap(jax.grad(f))          # gradient at many points (batched)
jax.value_and_grad(f)          # value and gradient together (training loops)
```

The key insight: these compose. `vmap(grad(f))` vectorizes the gradient computation — the same idiom appears in PINN training (gradient w.r.t. $x$ at many collocation points).

### Hessian → Normal Mode Frequencies

For a potential $V(\mathbf{r})$ at a minimum $\mathbf{r}_0$:

$$H_{ij} = \frac{\partial^2 V}{\partial r_i \partial r_j}\bigg|_{\mathbf{r}_0}$$

The Hessian is the **dynamical matrix** (assuming unit mass, $m=1$).
Its eigenvalues $\lambda_i$ give the squared normal mode frequencies: $\omega_i = \sqrt{\lambda_i}$.

For a non-quadratic potential, the Hessian changes with position — this is why we need autodiff rather than analytical shortcuts.


---

## Part 2: Physics-Informed Neural Networks (Theory)

### From Autodiff to PDE Solving

Autodiff differentiates **any** function w.r.t. its inputs — including a neural network.
If $u_\theta(x)$ is a network, autodiff gives $\partial u_\theta/\partial x$, $\partial^2 u_\theta/\partial x^2$, etc.
These are exactly the terms appearing in PDEs.

**PINNs** (Raissi et al., 2019): use a neural network as a **trial solution** and train it so the PDE is satisfied.

> Physics analogy: this is the **Ritz variational method** — minimize energy over a family of trial functions.
> PINNs use a neural network instead of a linear combination of basis functions.

### PINN Procedure

Given a boundary-value problem $\mathcal{D}[u](x) = f(x)$ on $\Omega$, with $u = g$ on $\partial\Omega$:

1. Parametrize: $u_\theta(x)$ (neural network)
2. Autodiff w.r.t. **input** $x$: compute $u'_\theta(x)$, $u''_\theta(x)$, …
3. Loss: how well does $u_\theta$ satisfy the PDE and BCs?
4. Backprop w.r.t. **weights** $\theta$: minimize the loss

Note the **dual role of autodiff**: differentiate w.r.t. inputs (step 2) AND w.r.t. weights (step 4).

### Loss Function

$$\mathcal{L} = \underbrace{\frac{1}{N}\sum_{i}\left(\mathcal{D}[u_\theta](x_i) - f(x_i)\right)^2}_{\mathcal{L}_{\text{pde}}} + w_{\text{bc}} \underbrace{\frac{1}{M}\sum_{j}\left(u_\theta(x_j^{\text{bc}}) - g(x_j^{\text{bc}})\right)^2}_{\mathcal{L}_{\text{bc}}}$$

**Collocation points** $\{x_i\}$: sampled inside the domain (no mesh needed).

### Eigenvalue Problems (Schrödinger Equation)

For the time-independent Schrödinger equation (atomic units, $\hbar = m = 1$):

$$-\frac{1}{2}\psi''(x) + V(x)\,\psi(x) = E\,\psi(x)$$

We add $E$ as a **trainable scalar** and augment the loss:

$$\mathcal{L} = \mathcal{L}_{\text{pde}} + w_{\text{norm}}\underbrace{\left(\int\psi^2\,dx - 1\right)^2}_{\text{normalization}} + w_{\text{bc}}\underbrace{\left[\psi(x_{\min})^2 + \psi(x_{\max})^2\right]}_{\text{boundary decay}}$$

> **Variational principle caveat** (§4 #22 fix): The PINN loss does **not** uniquely select
> the ground state. The training is biased toward the ground state only because we initialize
> $E_{\text{init}} < E_0$. If $E_{\text{init}} > E_0$, or when extending to a double-well
> potential, the network can silently converge to an excited state.
>
> To target an excited state $\psi_1$, add an **orthogonality loss**:
> $\mathcal{L}_{\text{orth}} = \left(\int \psi_1(x)\,\psi_0(x)\,dx\right)^2$
> where $\psi_0$ is a pre-trained frozen ground-state reference.

### PINN Inverse Problems

PINNs can also solve **inverse problems**: given noisy observations of $u(x)$, infer unknown parameters of the PDE.

For $-\frac{1}{2}\psi''(x) + (\frac{1}{2}\omega^2 x^2 + \lambda x^4)\psi = E\psi$ with unknown $\omega, \lambda$:

$$\mathcal{L}_{\text{inverse}} = \mathcal{L}_{\text{pde}} + w_{\text{norm}}\mathcal{L}_{\text{norm}} + w_{\text{bc}}\mathcal{L}_{\text{bc}} + w_{\text{data}} \underbrace{\frac{1}{N_{\text{obs}}}\sum_k \left(\psi_\theta(x_k) - \hat{\psi}_k\right)^2}_{\mathcal{L}_{\text{data}}}$$

Minimize over $(\theta, E, \omega, \lambda)$ simultaneously.

> **Forward pointer to L09**: Today's gradient-based calibration (Mission C) is a direct
> precursor to L09 Differentiable Physics, where we differentiate through an ODE integrator
> to recover force-field parameters from trajectories.

### PINN Limitations

- **Convergence**: slower than finite elements for standard well-posed problems
- **Loss balancing**: PDE and BC gradients can differ by orders of magnitude
- **Spectral bias**: networks learn low-frequency components first

---


## Mission A: JAX Autodiff for Physics

**Goal**: Become fluent with `jax.grad`, `jax.hessian`, and `jax.vmap` on physics functions.

| Step | Task | JAX function |
|------|------|-------------|
| 1 | Lennard-Jones potential: force & equilibrium | `jax.grad`, `jax.vmap` |
| 2 | Non-quadratic 2D potential: Hessian → normal modes | `jax.hessian` |
| 3 | Batch force computation | `jax.vmap(jax.grad(...))` |


### Step 1: Lennard-Jones Potential

$$V(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^6\right]$$

The force is $F(r) = -dV/dr$ and the equilibrium distance satisfies $F(r_{\text{eq}}) = 0$.

Analytically: $r_{\text{eq}} = 2^{1/6}\,\sigma \approx 1.1225$ (for $\sigma=1$).


In [ ]:
# --- Step 1: Lennard-Jones potential ---

def lj_potential(r: float, epsilon: float = 1.0, sigma: float = 1.0) -> float:
    """Lennard-Jones pair potential V(r)."""
    return 4.0 * epsilon * ((sigma / r) ** 12 - (sigma / r) ** 6)


# Force via autodiff: F(r) = -dV/dr
lj_force_grad = jax.grad(lj_potential)       # returns dV/dr
force_fn = lambda r: -lj_force_grad(r)       # negate for force

# Find equilibrium via gradient descent on V(r)
def find_equilibrium(r_init: float = 1.5, lr: float = 0.001, n_steps: int = 500) -> float:
    r = r_init
    for _ in range(n_steps):
        grad_v = jax.grad(lj_potential)(r)
        r = r - lr * grad_v
    return r


r_eq = find_equilibrium()
r_exact = 2.0 ** (1.0 / 6.0)  # sigma=1

print(f"Equilibrium (gradient descent): r_eq = {r_eq:.6f}")
print(f"Exact value (2^(1/6)):           r_eq = {r_exact:.6f}")
print(f"Error: {abs(r_eq - r_exact):.2e}")

# Plot V(r) and F(r)
r_arr = jnp.linspace(0.9, 3.0, 200)
v_arr = jax.vmap(lj_potential)(r_arr)
f_arr = jax.vmap(force_fn)(r_arr)

print(f"\nv_arr shape: {v_arr.shape}  (one potential value per r)")
print(f"f_arr shape: {f_arr.shape}  (one force value per r)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(r_arr, v_arr, "b-", lw=2)
ax1.axhline(0, color="gray", ls="--", alpha=0.5)
ax1.axvline(r_eq, color="r", ls="--", alpha=0.7, label=f"$r_{{eq}}={r_eq:.3f}$")
ax1.set_xlabel("$r$"); ax1.set_ylabel("$V(r)$")
ax1.set_title("Lennard-Jones Potential"); ax1.set_ylim(-1.5, 3); ax1.legend()

ax2.plot(r_arr, f_arr, "r-", lw=2)
ax2.axhline(0, color="gray", ls="--", alpha=0.5)
ax2.axvline(r_eq, color="b", ls="--", alpha=0.7, label=f"$r_{{eq}}={r_eq:.3f}$")
ax2.set_xlabel("$r$"); ax2.set_ylabel("$F(r) = -dV/dr$")
ax2.set_title("Force from Autodiff"); ax2.legend()
plt.tight_layout(); plt.show()

# Checkpoint
assert abs(r_eq - r_exact) < 0.01, f"Equilibrium distance off: {r_eq:.4f} vs {r_exact:.4f}"
print("\nCheckpoint PASSED: r_eq ≈ 2^(1/6)")


### Step 2: 2D Non-Quadratic Potential — Hessian & Normal Modes

We use a **Müller-Brown-inspired anisotropic double-well** potential (simplified for clarity):

$$V(x,y) = (x^2 - 1)^2 + 2y^2 + x\,y$$

This has two minima near $x \approx \pm 1$, $y \approx 0$ (shifted by the coupling term $xy$).
Unlike a pure quadratic, the Hessian **varies with position** — this is where autodiff
earns its keep.

The Hessian at a minimum gives the **dynamical matrix** (for unit mass $m=1$):
$$H_{ij} = \frac{\partial^2 V}{\partial r_i \partial r_j}$$

Eigenvalues $\lambda_i$ → normal mode frequencies $\omega_i = \sqrt{\lambda_i}$.


In [ ]:
# --- Step 2: 2D non-quadratic potential, Hessian, normal modes ---

def potential_2d(xy: jnp.ndarray) -> float:
    """2D double-well potential: V(x,y) = (x²-1)² + 2y² + xy.

    Has two minima near (±1, 0), shifted by coupling term.
    Non-quadratic: Hessian is position-dependent.
    """
    x, y = xy
    return (x**2 - 1.0)**2 + 2.0 * y**2 + x * y


grad_2d = jax.grad(potential_2d)
hessian_2d = jax.hessian(potential_2d)


# --- Find the two minima via gradient descent ---
def find_minimum(xy_init: jnp.ndarray, lr: float = 0.01, n_steps: int = 1000) -> jnp.ndarray:
    xy = xy_init
    for _ in range(n_steps):
        g = grad_2d(xy)
        xy = xy - lr * g
    return xy


min1 = find_minimum(jnp.array([ 1.0, 0.0]))
min2 = find_minimum(jnp.array([-1.0, 0.0]))

print("Minimum 1:", min1, "  V =", potential_2d(min1))
print("Minimum 2:", min2, "  V =", potential_2d(min2))

# --- Hessian and normal modes at each minimum ---
for label, pt in [("Minimum 1", min1), ("Minimum 2", min2)]:
    H = hessian_2d(pt)
    eigenvalues = jnp.linalg.eigvalsh(H)
    frequencies = jnp.sqrt(jnp.abs(eigenvalues))   # ω_i = sqrt(λ_i)
    print(f"\n{label}: {pt}")
    print(f"  Hessian:\n    {H[0]}\n    {H[1]}")
    print(f"  Eigenvalues (ω²):     {eigenvalues}")
    print(f"  Normal mode freqs ω:  {frequencies}")

# Verify: all eigenvalues positive (true minima, not saddle points)
H1 = hessian_2d(min1)
eigs1 = jnp.linalg.eigvalsh(H1)
assert jnp.all(eigs1 > 0), f"Not a minimum! Eigenvalues: {eigs1}"
print("\nCheckpoint PASSED: Hessian eigenvalues are both positive (true minimum).")

# --- Contour plot with minima and force vectors ---
x_grid = jnp.linspace(-2.0, 2.0, 120)
y_grid = jnp.linspace(-1.5, 1.5, 100)
XX, YY = jnp.meshgrid(x_grid, y_grid)
ZZ = jax.vmap(
    jax.vmap(lambda x, y: potential_2d(jnp.array([x, y])), in_axes=(None, 0)),
    in_axes=(0, None)
)(x_grid, y_grid).T

fig, ax = plt.subplots(figsize=(8, 5))
cs = ax.contourf(XX, YY, ZZ, levels=40, cmap="viridis")
plt.colorbar(cs, ax=ax, label="$V(x,y)$")
for pt, col, lbl in [(min1, "red", "min 1"), (min2, "cyan", "min 2")]:
    ax.plot(*pt, "o", color=col, markersize=9, label=lbl)
    g = grad_2d(pt)
    ax.quiver(*pt, *(-g * 0.1), color=col, scale=1, scale_units="xy",
              angles="xy", width=0.008)
ax.set_xlabel("$x$"); ax.set_ylabel("$y$")
ax.set_title("2D Double-Well: Minima + Force Vectors (F = -∇V)")
ax.legend(); plt.tight_layout(); plt.show()

print("\nNote: The two minima have DIFFERENT Hessians → different normal mode frequencies.")
print("This would not be visible with a purely quadratic potential.")


### Step 3: Batch Force Computation with `vmap`

`jax.vmap(jax.grad(f))` vectorizes the gradient over a batch of inputs.
This is the same pattern used in PINN training: compute $\psi''(x)$ at all collocation points at once.


In [ ]:
# --- Step 3: Batch force with vmap ---

batch_force = jax.vmap(lambda r: -jax.grad(lj_potential)(r))

r_batch = jnp.linspace(1.0, 3.0, 10)
forces = batch_force(r_batch)

print(f"r_batch shape: {r_batch.shape}")
print(f"forces shape:  {forces.shape}")
print()
print(f"{'r':>8s}  {'F(r)':>12s}")
print(f"{'─'*8}  {'─'*12}")
for rv, fv in zip(r_batch, forces):
    print(f"{rv:8.4f}  {fv:12.6f}")

print("\nCheckpoint: Force at r_eq ≈ 0")
f_at_eq = batch_force(jnp.array([r_exact]))[0]
print(f"  F({r_exact:.4f}) = {f_at_eq:.6f}  (should be ≈ 0)")
assert abs(f_at_eq) < 0.01
print("Checkpoint PASSED")


---

## Mission B: PINN for the Quantum Harmonic Oscillator (QHO)

**Problem** (atomic units, $\hbar = m = 1$):

$$-\frac{1}{2}\psi''(x) + \frac{1}{2}x^2\,\psi(x) = E\,\psi(x)$$

**Exact ground state**: $E_0 = 0.5$, $\psi_0(x) = \pi^{-1/4}\exp(-x^2/2)$

We know the answer — so we can verify the PINN works before applying it where analytic solutions don't exist.

### What you implement (3 TODOs):
1. `psi_xx_fn(params, x)` — second derivative $\psi''(x)$ via autodiff
2. `pinn_loss(params, E, x_grid)` — PDE residual + normalization + BC
3. `train_step(...)` — gradient update for both `params` and `E`

### Scaffolding provided:
- MLP definition (tanh activations, He init)
- Training loop + plotting
- Exact solution for comparison

> **Variational principle note** (§4 #22): Starting with $E_{\text{init}} = 0.3 < E_0 = 0.5$
> biases training toward the ground state. This is intentional here, but when extending
> to a double-well or targeting an excited state, this bias can cause **silent wrong convergence**.
> See Mission B-ext2 for the fix (orthogonality loss).


In [ ]:
# --- PINN infrastructure (shared by Missions B, B-ext1, B-ext2, C) ---

# Hyperparameters
HIDDEN_DIM = 64
N_LAYERS = 3
X_MIN, X_MAX = -5.0, 5.0
N_COLLOC = 300

# ── MLP (pure JAX, no Flax) ──────────────────────────────────────────────────

def init_mlp(key: jax.Array, layer_sizes: list) -> list:
    """Initialize MLP parameters: list of (W, b) tuples.

    Uses He initialization (scale = sqrt(2/fan_in)) appropriate for tanh
    at small hidden sizes. Atomic units: network maps scalar x -> scalar psi(x).
    """
    params = []
    for in_dim, out_dim in zip(layer_sizes[:-1], layer_sizes[1:]):
        key, subkey = jax.random.split(key)
        scale = jnp.sqrt(2.0 / in_dim)
        W = scale * jax.random.normal(subkey, (in_dim, out_dim))
        b = jnp.zeros(out_dim)
        params.append((W, b))
    return params


def mlp_forward(params: list, x_scalar: float) -> float:
    """Forward pass: scalar x → scalar output (ψ value)."""
    h = jnp.array([x_scalar])   # shape (1,)
    for W, b in params[:-1]:
        h = jnp.tanh(h @ W + b)
    W_last, b_last = params[-1]
    return jnp.squeeze(h @ W_last + b_last)


def psi_fn(params: list, x: float) -> float:
    """ψ(x): neural network output."""
    return mlp_forward(params, x)


# --- Initialize a default network ---
key, subkey = jax.random.split(key)
layer_sizes = [1] + [HIDDEN_DIM] * N_LAYERS + [1]
params_init = init_mlp(subkey, layer_sizes)

# Shape check
x_test = 0.0
psi_test = psi_fn(params_init, x_test)
print(f"psi_fn output shape: {jnp.shape(psi_test)}  (scalar, as expected)")

x_grid = jnp.linspace(X_MIN, X_MAX, N_COLLOC)
psi_batch = jax.vmap(psi_fn, in_axes=(None, 0))(params_init, x_grid)
print(f"Batched psi over grid: {psi_batch.shape}  (N_COLLOC={N_COLLOC} values)")


### TODO 1: Second Derivative `psi_xx_fn`

Implement $\psi''(x)$ using `jax.grad` twice with `argnums=1`.

```python
# Hint:
#   dpsi_dx  = jax.grad(psi_fn, argnums=1)       # ψ'(x)
#   d2psi_dx2 = jax.grad(dpsi_dx, argnums=1)     # ψ''(x)
```


In [ ]:
# TODO 1 — Implement psi_xx_fn
# Replace the pass with the correct implementation.

def psi_xx_fn(params: list, x: float) -> float:
    """ψ''(x): second derivative of ψ w.r.t. x via autodiff."""
    dpsi_dx = jax.grad(psi_fn, argnums=1)
    d2psi_dx2 = jax.grad(dpsi_dx, argnums=1)
    return d2psi_dx2(params, x)


# Self-test: for a Gaussian, ψ(x)=exp(-x²/2), ψ''(x)=(x²-1)exp(-x²/2)
def test_gaussian(x):
    return jnp.exp(-0.5 * x**2)

# We need params that just return the Gaussian — use a lambda trick for the test
class _GaussianNet:
    def __call__(self, params, x):
        return jnp.exp(-0.5 * x**2)

_g = _GaussianNet()
# Instead, test with numerical comparison on the actual MLP
x_check = 1.0
psi_xx_val = psi_xx_fn(params_init, x_check)
print(f"ψ''({x_check}) from autodiff: {psi_xx_val:.6f}  (shape: {jnp.shape(psi_xx_val)})")

# Batch second derivative over collocation grid
psi_xx_batch = jax.vmap(psi_xx_fn, in_axes=(None, 0))(params_init, x_grid)
print(f"Batched ψ'' shape: {psi_xx_batch.shape}")
print("TODO 1 complete.")


### TODO 2: PINN Loss Function

Implement `pinn_loss(params, E, x_grid)`:

$$\mathcal{L} = \mathcal{L}_{\text{pde}} + 10\,\mathcal{L}_{\text{norm}} + 5\,\mathcal{L}_{\text{bc}}$$

$$\mathcal{L}_{\text{pde}} = \frac{1}{N}\sum_i \left(-\frac{1}{2}\psi''(x_i) + \frac{1}{2}x_i^2\psi(x_i) - E\psi(x_i)\right)^2$$

$$\mathcal{L}_{\text{norm}} = \left(\int \psi(x)^2\,dx - 1\right)^2 \approx \left(\text{trapz}(\psi^2, x) - 1\right)^2$$

$$\mathcal{L}_{\text{bc}} = \psi(x_{\min})^2 + \psi(x_{\max})^2$$


In [ ]:
# TODO 2 — Implement pinn_loss for QHO

def pinn_loss(params: list, E: float, x_grid: jnp.ndarray) -> float:
    """Total PINN loss for the Schrödinger equation.

    Atomic units: ℏ = m = 1.  V(x) = ½ x².
    Loss = L_pde + 10 * L_norm + 5 * L_bc
    """
    # Vectorize ψ and ψ'' over collocation grid
    psi_vals   = jax.vmap(psi_fn,    in_axes=(None, 0))(params, x_grid)
    psi_xx_vals = jax.vmap(psi_xx_fn, in_axes=(None, 0))(params, x_grid)

    # PDE residual: -ψ''/2 + x²ψ/2 - Eψ = 0
    residual = -0.5 * psi_xx_vals + 0.5 * x_grid**2 * psi_vals - E * psi_vals
    loss_pde = jnp.mean(residual**2)

    # Normalization: ∫ψ² dx = 1  (trapezoidal rule)
    integral_psi2 = jnp.trapezoid(psi_vals**2, x_grid)
    loss_norm = (integral_psi2 - 1.0)**2

    # Boundary conditions: ψ vanishes at domain edges
    loss_bc = psi_vals[0]**2 + psi_vals[-1]**2

    return loss_pde + 10.0 * loss_norm + 5.0 * loss_bc


# Test the loss
E_init = 0.3   # intentionally below E_0=0.5 to bias toward ground state
loss_val = pinn_loss(params_init, E_init, x_grid)
print(f"Initial loss: {loss_val:.6f}  (should be large, we haven't trained yet)")
print("TODO 2 complete.")


### TODO 3: Training Step

Implement `train_step` using `jax.value_and_grad` with `argnums=(0, 1)` to update
both `params` (network weights) and `E` (energy eigenvalue) simultaneously.


In [ ]:
# TODO 3 — Implement train_step

def train_step(params, E, opt_state_params, opt_state_E, x_grid,
               optimizer_params, optimizer_E):
    """Single training step: compute gradients and update params + E."""
    loss, (grad_params, grad_E) = jax.value_and_grad(
        pinn_loss, argnums=(0, 1)
    )(params, E, x_grid)

    updates_p, opt_state_params = optimizer_params.update(grad_params, opt_state_params)
    params = optax.apply_updates(params, updates_p)

    updates_E, opt_state_E = optimizer_E.update(grad_E, opt_state_E)
    E = optax.apply_updates(E, updates_E)

    return params, E, opt_state_params, opt_state_E, loss


# --- Training QHO PINN ---
LEARNING_RATE = 1e-3
N_STEPS = 8000
PRINT_EVERY = 1000

key, subkey = jax.random.split(key)
params_b = init_mlp(subkey, layer_sizes)
E_b = jnp.array(0.3)   # E_init < E_0=0.5  →  ground state bias (intentional)

optimizer_params = optax.adam(LEARNING_RATE)
optimizer_E      = optax.adam(LEARNING_RATE)
opt_state_p = optimizer_params.init(params_b)
opt_state_E = optimizer_E.init(E_b)

# JIT compile
train_step_jit = jax.jit(train_step, static_argnums=(5, 6))

loss_hist, E_hist = [], []
print("Training QHO PINN (Mission B)...")
for step in range(N_STEPS):
    params_b, E_b, opt_state_p, opt_state_E, loss = train_step_jit(
        params_b, E_b, opt_state_p, opt_state_E, x_grid,
        optimizer_params, optimizer_E,
    )
    loss_hist.append(float(loss))
    E_hist.append(float(E_b))
    if step % PRINT_EVERY == 0 or step == N_STEPS - 1:
        print(f"  Step {step:5d} | Loss = {loss:.6f} | E = {float(E_b):.6f}")

print(f"\nFinal E = {float(E_b):.6f}  (exact E_0 = 0.5)")
print(f"Error |E - 0.5| = {abs(float(E_b) - 0.5):.4f}")


In [ ]:
# --- Plot Mission B results ---

def psi_exact_ground(x):
    """Exact QHO ground state: ψ₀(x) = π^(-1/4) exp(-x²/2)."""
    return jnp.pi**(-0.25) * jnp.exp(-0.5 * x**2)


x_plot = jnp.linspace(X_MIN, X_MAX, 500)
psi_nn = jax.vmap(psi_fn, in_axes=(None, 0))(params_b, x_plot)
psi_ex = psi_exact_ground(x_plot)

# Fix sign ambiguity (global phase)
if jnp.sum(psi_nn * psi_ex) < 0:
    psi_nn = -psi_nn

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(x_plot, psi_ex, "b-", lw=2, label="Exact $\psi_0(x)$")
axes[0].plot(x_plot, psi_nn, "r--", lw=2, label="PINN $\psi_\theta(x)$")
axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$\psi(x)$")
axes[0].set_title("Wavefunction"); axes[0].legend()

axes[1].plot(E_hist, "g-", lw=1)
axes[1].axhline(0.5, color="k", ls="--", alpha=0.7, label="$E_0=0.5$")
axes[1].set_xlabel("Step"); axes[1].set_ylabel("$E$")
axes[1].set_title("Energy Convergence"); axes[1].legend()

axes[2].semilogy(loss_hist, "m-", lw=1)
axes[2].set_xlabel("Step"); axes[2].set_ylabel("Loss")
axes[2].set_title("Training Loss")

plt.tight_layout(); plt.show()

# Checkpoint
assert abs(float(E_b) - 0.5) < 0.05, f"Energy did not converge: E={float(E_b):.4f}"
print(f"Checkpoint PASSED: E converged to {float(E_b):.4f} (target 0.5)")


---

## Mission B-ext1 (Graded): Anharmonic Potential PINN

**Potential**:

$$V(x) = \frac{1}{2}x^2 + \lambda x^4, \qquad \lambda = 0.1$$

The Schrödinger equation becomes:

$$-\frac{1}{2}\psi''(x) + \left(\frac{1}{2}x^2 + \lambda x^4\right)\psi(x) = E\,\psi(x)$$

For $\lambda = 0.1$, perturbation theory gives $E_0 \approx 0.5 + \frac{3\lambda}{4} = 0.575$.

**Task**: Adapt the QHO PINN by replacing $V(x) = \frac{1}{2}x^2$ with the anharmonic potential.
Verify that $E_0 > 0.5$ (energy increases with $\lambda$).


In [ ]:
# --- Mission B-ext1: Anharmonic PINN ---

LAMBDA = 0.1   # anharmonicity parameter


def pinn_loss_anharmonic(params: list, E: float, x_grid: jnp.ndarray,
                          lam: float = LAMBDA) -> float:
    """PINN loss for anharmonic potential V(x) = ½x² + λx⁴.

    Only the PDE residual changes; normalization and BC are identical to QHO.
    Atomic units: ℏ = m = 1.
    """
    psi_vals    = jax.vmap(psi_fn,    in_axes=(None, 0))(params, x_grid)
    psi_xx_vals = jax.vmap(psi_xx_fn, in_axes=(None, 0))(params, x_grid)

    # Anharmonic PDE residual: -ψ''/2 + (x²/2 + λx⁴)ψ - Eψ = 0
    V_x = 0.5 * x_grid**2 + lam * x_grid**4
    residual = -0.5 * psi_xx_vals + V_x * psi_vals - E * psi_vals
    loss_pde = jnp.mean(residual**2)

    integral_psi2 = jnp.trapezoid(psi_vals**2, x_grid)
    loss_norm = (integral_psi2 - 1.0)**2
    loss_bc = psi_vals[0]**2 + psi_vals[-1]**2

    return loss_pde + 10.0 * loss_norm + 5.0 * loss_bc


def train_step_gen(params, E, opt_state_params, opt_state_E, x_grid,
                   optimizer_params, optimizer_E, loss_fn):
    """Generic training step accepting any loss function."""
    loss, (grad_params, grad_E) = jax.value_and_grad(
        loss_fn, argnums=(0, 1)
    )(params, E, x_grid)

    updates_p, opt_state_params = optimizer_params.update(grad_params, opt_state_params)
    params = optax.apply_updates(params, updates_p)
    updates_E, opt_state_E = optimizer_E.update(grad_E, opt_state_E)
    E = optax.apply_updates(E, updates_E)
    return params, E, opt_state_params, opt_state_E, loss


# --- Train anharmonic PINN ---
key, subkey = jax.random.split(key)
params_anh = init_mlp(subkey, layer_sizes)
E_anh = jnp.array(0.3)

opt_p_anh = optax.adam(LEARNING_RATE).init(params_anh)
opt_E_anh = optax.adam(LEARNING_RATE).init(E_anh)
opt_params_anh = optax.adam(LEARNING_RATE)
opt_E_obj_anh  = optax.adam(LEARNING_RATE)

loss_hist_anh, E_hist_anh = [], []
print(f"Training anharmonic PINN (λ={LAMBDA})...")
for step in range(N_STEPS):
    params_anh, E_anh, opt_p_anh, opt_E_anh, loss = train_step_gen(
        params_anh, E_anh, opt_p_anh, opt_E_anh, x_grid,
        opt_params_anh, opt_E_obj_anh, pinn_loss_anharmonic,
    )
    loss_hist_anh.append(float(loss))
    E_hist_anh.append(float(E_anh))
    if step % PRINT_EVERY == 0 or step == N_STEPS - 1:
        print(f"  Step {step:5d} | Loss = {loss:.6f} | E = {float(E_anh):.6f}")

E0_perturbation = 0.5 + 0.75 * LAMBDA   # first-order PT: E_0 ≈ 0.5 + (3λ/4)
print(f"\nFinal E (anharmonic) = {float(E_anh):.6f}")
print(f"Perturbation theory:   E_0 ≈ {E0_perturbation:.4f}")
print(f"(Exact numerical from diagonalization ≈ 0.5593 for λ=0.1)")
assert float(E_anh) > 0.5, "Anharmonic E_0 should be > 0.5 (positive λ raises energy)"
print("\nCheckpoint PASSED: E_0(anharmonic) > E_0(QHO)")

# Plot comparison
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(E_hist,     label=f"QHO (λ=0)")
ax.plot(E_hist_anh, label=f"Anharmonic (λ={LAMBDA})")
ax.axhline(0.5,             color="blue",  ls="--", alpha=0.7, label="$E_0^{QHO}=0.5$")
ax.axhline(E0_perturbation, color="orange", ls="--", alpha=0.7,
           label=f"PT: $E_0 \approx {E0_perturbation:.3f}$")
ax.set_xlabel("Step"); ax.set_ylabel("$E$")
ax.set_title("Energy Convergence: QHO vs Anharmonic"); ax.legend()
plt.tight_layout(); plt.show()


---

## Mission B-ext2 (Graded): Excited State PINN with Orthogonality Loss

**Problem**: Find the first excited state $\psi_1(x)$ of the QHO, with $E_1 = 1.5$.

The variational principle alone **cannot** distinguish $\psi_1$ from $\psi_0$ without a
constraint. We add an **orthogonality loss** using the frozen ground-state network:

$$\mathcal{L}_{\text{orth}} = \left(\int \psi_1(x)\,\psi_0^*(x)\,dx\right)^2$$

This forces $\psi_1 \perp \psi_0$, as required by the eigenvalue structure of the Hamiltonian.

> **Key insight**: Without $\mathcal{L}_{\text{orth}}$, initializing $E_{\text{init}} > E_0$
> does NOT reliably converge to $E_1$. The network can find distorted versions of $\psi_0$
> with energies between $E_0$ and $E_1$ that have lower PDE residual.
> This is the §4 #22 "silent wrong convergence" issue.


In [ ]:
# --- Mission B-ext2: Excited-state PINN ---

# Freeze the trained ground-state params from Mission B
params_gs_frozen = params_b   # ground state reference

def psi_exact_excited(x):
    """Exact QHO first excited state: ψ₁(x) = (2/π)^(1/4) · x · exp(-x²/2)."""
    return (2.0 / jnp.pi)**0.25 * x * jnp.exp(-0.5 * x**2)


def pinn_loss_excited(params: list, E: float, x_grid: jnp.ndarray,
                       params_gs: list, w_orth: float = 20.0) -> float:
    """PINN loss for the first excited state of QHO.

    Adds orthogonality constraint w.r.t. frozen ground-state reference ψ₀.
    Atomic units: ℏ = m = 1.  V(x) = ½ x².
    """
    psi_vals    = jax.vmap(psi_fn,    in_axes=(None, 0))(params,    x_grid)
    psi_xx_vals = jax.vmap(psi_xx_fn, in_axes=(None, 0))(params,    x_grid)
    psi_gs_vals = jax.vmap(psi_fn,    in_axes=(None, 0))(params_gs, x_grid)

    # PDE residual (same Hamiltonian)
    residual = -0.5 * psi_xx_vals + 0.5 * x_grid**2 * psi_vals - E * psi_vals
    loss_pde = jnp.mean(residual**2)

    # Normalization
    loss_norm = (jnp.trapezoid(psi_vals**2, x_grid) - 1.0)**2

    # Boundary conditions
    loss_bc = psi_vals[0]**2 + psi_vals[-1]**2

    # Orthogonality: <ψ₁|ψ₀> = 0
    overlap = jnp.trapezoid(psi_vals * psi_gs_vals, x_grid)
    loss_orth = overlap**2

    return loss_pde + 10.0 * loss_norm + 5.0 * loss_bc + w_orth * loss_orth


# --- Train excited-state PINN (higher E_init to target E_1=1.5) ---
key, subkey = jax.random.split(key)
params_ex = init_mlp(subkey, layer_sizes)
E_ex = jnp.array(1.0)   # start between E_0 and E_1

opt_p_ex_obj = optax.adam(LEARNING_RATE)
opt_E_ex_obj = optax.adam(LEARNING_RATE)
opt_p_ex = opt_p_ex_obj.init(params_ex)
opt_E_ex = opt_E_ex_obj.init(E_ex)

N_STEPS_EX = 10000
loss_hist_ex, E_hist_ex = [], []
print("Training excited-state PINN (with orthogonality loss)...")
for step in range(N_STEPS_EX):
    loss_ex, (gp, gE) = jax.value_and_grad(
        pinn_loss_excited, argnums=(0, 1)
    )(params_ex, E_ex, x_grid, params_gs_frozen)

    updates_p, opt_p_ex = opt_p_ex_obj.update(gp, opt_p_ex)
    params_ex = optax.apply_updates(params_ex, updates_p)
    updates_E, opt_E_ex = opt_E_ex_obj.update(gE, opt_E_ex)
    E_ex = optax.apply_updates(E_ex, updates_E)

    loss_hist_ex.append(float(loss_ex))
    E_hist_ex.append(float(E_ex))
    if step % 2000 == 0 or step == N_STEPS_EX - 1:
        print(f"  Step {step:5d} | Loss = {loss_ex:.6f} | E = {float(E_ex):.6f}")

print(f"\nFinal E₁ = {float(E_ex):.6f}  (exact E₁ = 1.5)")
print(f"Error |E₁ - 1.5| = {abs(float(E_ex) - 1.5):.4f}")

# Plot
x_plot = jnp.linspace(X_MIN, X_MAX, 500)
psi_nn_ex = jax.vmap(psi_fn, in_axes=(None, 0))(params_ex, x_plot)
psi_ex_exact = psi_exact_excited(x_plot)

if jnp.sum(psi_nn_ex * psi_ex_exact) < 0:
    psi_nn_ex = -psi_nn_ex

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_plot, psi_ex_exact, "b-",  lw=2, label="Exact $\psi_1(x)$")
axes[0].plot(x_plot, psi_nn_ex,    "r--", lw=2, label="PINN $\psi_{\theta}(x)$")
axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$\psi_1(x)$")
axes[0].set_title("First Excited State"); axes[0].legend()

axes[1].plot(E_hist_ex, "g-", lw=1)
axes[1].axhline(1.5, color="k", ls="--", alpha=0.7, label="$E_1=1.5$")
axes[1].axhline(0.5, color="b", ls=":", alpha=0.5, label="$E_0=0.5$")
axes[1].set_xlabel("Step"); axes[1].set_ylabel("$E$")
axes[1].set_title("Excited State Energy Convergence"); axes[1].legend()
plt.tight_layout(); plt.show()

# Checkpoint
assert abs(float(E_ex) - 1.5) < 0.1, f"Excited state energy did not converge: E={float(E_ex):.4f}"
print(f"Checkpoint PASSED: E₁ converged to {float(E_ex):.4f} (target 1.5)")


---

## Mission C (New): PINN Inverse Problem — Recover Unknown Potential Coefficients

**Setup**: We observe $N_{\text{obs}} = 20$ noisy measurements of $\psi(x)$ at scattered points.
The true wavefunction comes from the anharmonic QHO with $\omega=1$, $\lambda=0.1$.
We do **not** know $\omega$ and $\lambda$ — we want to infer them from the data.

$$V(x; \omega, \lambda) = \frac{1}{2}\omega^2 x^2 + \lambda x^4$$

We treat $\omega$ and $\lambda$ as **trainable scalars**, jointly optimized with the network
weights $\theta$ and energy $E$:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{pde}} + w_{\text{norm}}\mathcal{L}_{\text{norm}} + w_{\text{bc}}\mathcal{L}_{\text{bc}} + w_{\text{data}}\mathcal{L}_{\text{data}}$$

$$\mathcal{L}_{\text{data}} = \frac{1}{N_{\text{obs}}}\sum_k \left(\psi_\theta(x_k) - \hat{\psi}_k\right)^2$$

> **Physics analogy**: this is **data assimilation** — using a physics model + sparse
> noisy observations to infer unknown parameters of the governing equation.
>
> **Forward pointer to L09**: In L09, we differentiate through an ODE integrator to recover
> force-field parameters from molecular dynamics trajectories. Same principle — gradient-based
> calibration of physics model parameters — but applied to a dynamical (time-dependent) system.

### TODO 1: Set up trainable potential parameters and data-fit loss
### TODO 2: `jax.value_and_grad` with `argnums=(0, 1, 2, 3)` for joint optimization
### TODO 3: Plot recovered $\omega$, $\lambda$ vs true values


In [ ]:
# --- Mission C: PINN Inverse Problem ---

# True parameters (unknown to the model — we only see noisy observations)
OMEGA_TRUE  = 1.0
LAMBDA_TRUE = 0.1
N_OBS = 20
SIGMA_NOISE = 0.01

# Generate "experimental" observations from anharmonic PINN solution
key, subkey = jax.random.split(key)
x_obs = jnp.sort(jax.random.uniform(subkey, (N_OBS,), minval=-3.0, maxval=3.0))
psi_obs_clean = jax.vmap(psi_fn, in_axes=(None, 0))(params_anh, x_obs)

# Fix sign so observations match positive-lobe convention
if jnp.sum(psi_obs_clean) < 0:
    psi_obs_clean = -psi_obs_clean

key, subkey = jax.random.split(key)
noise = SIGMA_NOISE * jax.random.normal(subkey, (N_OBS,))
psi_obs = psi_obs_clean + noise

print(f"Observations: {N_OBS} points, noise σ={SIGMA_NOISE}")
print(f"True parameters: ω={OMEGA_TRUE:.2f}, λ={LAMBDA_TRUE:.3f}")
print(f"x_obs shape: {x_obs.shape}, psi_obs shape: {psi_obs.shape}")


In [ ]:
# TODO 1 — PINN inverse-problem loss

def pinn_loss_inverse(
    params: list,
    E: float,
    omega: float,
    lam: float,
    x_grid: jnp.ndarray,
    x_obs: jnp.ndarray,
    psi_obs: jnp.ndarray,
    w_data: float = 100.0,
) -> float:
    """PINN loss for inverse problem: recover ω, λ from noisy observations.

    Parameters
    ----------
    params  : network weights θ
    E       : trainable energy eigenvalue
    omega   : trainable potential parameter ω (true: 1.0)
    lam     : trainable anharmonicity λ (true: 0.1)
    x_grid  : collocation points for PDE residual
    x_obs   : observation locations
    psi_obs : noisy wavefunction observations
    w_data  : data-fit loss weight
    """
    psi_vals    = jax.vmap(psi_fn,    in_axes=(None, 0))(params, x_grid)
    psi_xx_vals = jax.vmap(psi_xx_fn, in_axes=(None, 0))(params, x_grid)

    # PDE residual with unknown V(x; ω, λ) = ½ω²x² + λx⁴
    V_x = 0.5 * omega**2 * x_grid**2 + lam * x_grid**4
    residual = -0.5 * psi_xx_vals + V_x * psi_vals - E * psi_vals
    loss_pde = jnp.mean(residual**2)

    # Normalization
    loss_norm = (jnp.trapezoid(psi_vals**2, x_grid) - 1.0)**2

    # Boundary conditions
    loss_bc = psi_vals[0]**2 + psi_vals[-1]**2

    # TODO 1: Data-fit term — evaluate ψ_θ at observation locations
    # psi_at_obs = ?  (hint: jax.vmap(psi_fn, ...) on x_obs)
    # loss_data  = ?  (hint: mean squared error against psi_obs)
    psi_at_obs = jax.vmap(psi_fn, in_axes=(None, 0))(params, x_obs)
    loss_data  = jnp.mean((psi_at_obs - psi_obs)**2)

    return loss_pde + 10.0 * loss_norm + 5.0 * loss_bc + w_data * loss_data


In [ ]:
# TODO 2 — Train inverse PINN: joint optimization of (θ, E, ω, λ)

key, subkey = jax.random.split(key)
params_inv = init_mlp(subkey, layer_sizes)
E_inv   = jnp.array(0.3)
omega_inv = jnp.array(1.5)   # intentionally wrong initial guess
lam_inv   = jnp.array(0.3)   # intentionally wrong initial guess

opt_params_inv = optax.adam(1e-3)
opt_E_inv      = optax.adam(1e-3)
opt_omega_inv  = optax.adam(5e-3)
opt_lam_inv    = optax.adam(5e-3)

opt_state_p_inv = opt_params_inv.init(params_inv)
opt_state_E_inv = opt_E_inv.init(E_inv)
opt_state_w_inv = opt_omega_inv.init(omega_inv)
opt_state_l_inv = opt_lam_inv.init(lam_inv)

N_STEPS_INV = 10000
omega_hist, lam_hist, E_inv_hist, loss_inv_hist = [], [], [], []

print(f"Training inverse PINN...")
print(f"Initial guess: ω={float(omega_inv):.2f}, λ={float(lam_inv):.3f}")
print(f"True values:   ω={OMEGA_TRUE:.2f}, λ={LAMBDA_TRUE:.3f}")

for step in range(N_STEPS_INV):
    # TODO 2: jax.value_and_grad with argnums=(0,1,2,3) for all trainable quantities
    loss_inv, (gp, gE, gomega, glam) = jax.value_and_grad(
        pinn_loss_inverse, argnums=(0, 1, 2, 3)
    )(params_inv, E_inv, omega_inv, lam_inv, x_grid, x_obs, psi_obs)

    updates_p, opt_state_p_inv = opt_params_inv.update(gp,     opt_state_p_inv)
    updates_E, opt_state_E_inv = opt_E_inv.update(gE,          opt_state_E_inv)
    updates_w, opt_state_w_inv = opt_omega_inv.update(gomega,  opt_state_w_inv)
    updates_l, opt_state_l_inv = opt_lam_inv.update(glam,      opt_state_l_inv)

    params_inv = optax.apply_updates(params_inv, updates_p)
    E_inv      = optax.apply_updates(E_inv,      updates_E)
    omega_inv  = optax.apply_updates(omega_inv,  updates_w)
    lam_inv    = optax.apply_updates(lam_inv,    updates_l)

    omega_hist.append(float(omega_inv))
    lam_hist.append(float(lam_inv))
    E_inv_hist.append(float(E_inv))
    loss_inv_hist.append(float(loss_inv))

    if step % 2000 == 0 or step == N_STEPS_INV - 1:
        print(f"  Step {step:5d} | Loss={loss_inv:.5f} | E={float(E_inv):.4f} | "
              f"ω={float(omega_inv):.4f} | λ={float(lam_inv):.4f}")

print(f"\nRecovered: ω = {float(omega_inv):.4f}  (true: {OMEGA_TRUE})")
print(f"Recovered: λ = {float(lam_inv):.4f}  (true: {LAMBDA_TRUE})")
print(f"Error |ω_rec - ω_true| = {abs(float(omega_inv) - OMEGA_TRUE):.4f}")
print(f"Error |λ_rec - λ_true| = {abs(float(lam_inv) - LAMBDA_TRUE):.4f}")


In [ ]:
# TODO 3 — Plot recovered parameters vs true values

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Parameter recovery trajectories
axes[0].plot(omega_hist, label="ω recovered")
axes[0].axhline(OMEGA_TRUE, color="r", ls="--", label=f"ω true = {OMEGA_TRUE}")
axes[0].set_xlabel("Step"); axes[0].set_title("ω Recovery")
axes[0].legend()

axes[1].plot(lam_hist, label="λ recovered")
axes[1].axhline(LAMBDA_TRUE, color="r", ls="--", label=f"λ true = {LAMBDA_TRUE}")
axes[1].set_xlabel("Step"); axes[1].set_title("λ Recovery")
axes[1].legend()

# Wavefunction at recovered parameters
x_plot = jnp.linspace(X_MIN, X_MAX, 500)
psi_inv_plot = jax.vmap(psi_fn, in_axes=(None, 0))(params_inv, x_plot)
psi_ref_plot = jax.vmap(psi_fn, in_axes=(None, 0))(params_anh,  x_plot)
if jnp.sum(psi_inv_plot) < 0: psi_inv_plot = -psi_inv_plot
if jnp.sum(psi_ref_plot) < 0: psi_ref_plot = -psi_ref_plot

axes[2].plot(x_plot, psi_ref_plot, "b-",  lw=2, label="Reference (Mission B-ext1)")
axes[2].plot(x_plot, psi_inv_plot, "r--", lw=2, label="Inverse PINN reconstruction")
axes[2].scatter(x_obs, psi_obs, s=30, c="k", zorder=5, label="Noisy observations")
axes[2].set_xlabel("$x$"); axes[2].set_ylabel("$\psi(x)$")
axes[2].set_title(f"Wavefunction Reconstruction\n(ω={float(omega_inv):.3f}, λ={float(lam_inv):.4f})")
axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

# Checkpoint
assert abs(float(omega_inv) - OMEGA_TRUE)  < 0.1, f"ω not recovered: {float(omega_inv):.4f}"
assert abs(float(lam_inv)   - LAMBDA_TRUE) < 0.05, f"λ not recovered: {float(lam_inv):.4f}"
print(f"\nCheckpoint PASSED: |ω_rec - 1.0| < 0.1, |λ_rec - 0.1| < 0.05")


---

## Summary

| Mission | Task | Key JAX | Physics |
|---------|------|---------|---------|
| A-1 | LJ potential force & equilibrium | `jax.grad` | $F = -dV/dr$ |
| A-2 | 2D non-quadratic Hessian | `jax.hessian` | Normal mode frequencies |
| A-3 | Batch force | `jax.vmap(jax.grad(...))` | Vectorized computation |
| B | QHO PINN ground state | `value_and_grad`, `argnums` | Schrödinger eq., $E_0=0.5$ |
| B-ext1 | Anharmonic PINN | Modified loss | Perturbative correction $E_0 > 0.5$ |
| B-ext2 | Excited state + orthogonality | Custom loss | $E_1=1.5$, $\psi_1 \perp \psi_0$ |
| C | Inverse problem | `argnums=(0,1,2,3)` | Recover $\omega, \lambda$ from data |

### Key Takeaways

1. **Autodiff is exact**: same floating-point operations as forward eval, no step-size tuning.

2. **Dual role in PINNs**: differentiate w.r.t. inputs (to evaluate PDE terms) AND w.r.t.
   weights (backprop for training). Two different `argnums`, one unified framework.

3. **Eigenvalue bias** (§4 #22): The variational PINN does not uniquely select the ground state.
   $E_{\text{init}} < E_0$ provides ground-state bias. For excited states or double-well
   potentials, add an orthogonality loss — otherwise training can silently converge wrong.

4. **Inverse problems**: Making $\omega$, $\lambda$ trainable scalars and adding a data-fit term
   turns the forward PINN into a parameter estimation engine. This is **data assimilation** in
   physics parlance.

### Forward Pointer to L09 (Differentiable Physics)

Today's Mission C is a **static** inverse problem: we recovered $\omega$, $\lambda$ from
wavefunction snapshots. In L09, we apply the same gradient-based calibration idea to
**dynamical** systems: differentiate through an ODE integrator (Verlet/leapfrog) to recover
force-field parameters from molecular dynamics trajectories. The key difference is that
the "PDE" is now a time-dependent ODE and we backpropagate through time steps.

### References

1. Raissi, Perdikaris, Karniadakis. "Physics-informed neural networks." *J. Comput. Phys.* **378**, 2019. [DOI:10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045)
2. Baydin, Pearlmutter, Radul, Siskind. "Automatic Differentiation in ML: a Survey." *JMLR* **18**(153), 2018. [arXiv:1502.05767](https://arxiv.org/abs/1502.05767)
3. Jin, Mattheakis, Protopapas. "PINNs for Quantum Eigenvalue Problems." 2022. [arXiv:2203.00451](https://arxiv.org/abs/2203.00451)
4. Wang, Teng, Perdikaris. "Understanding and mitigating gradient flow pathologies in PINNs." *SIAM J. Sci. Comput.* **43**(5), 2021. [arXiv:2001.04536](https://arxiv.org/abs/2001.04536)
5. Olah, C. "Calculus on Computational Graphs: Backpropagation." 2015. [colah.github.io](https://colah.github.io/posts/2015-08-Backprop/)
